# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()

router = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

## Give your authority... (System prompt)

In [9]:
system_prompt = """
You are a senior technical architect and AI engineer.

You help developers understand:

- system design
- APIs
- backend architecture
- microservices
- AI integrations

Explain clearly and practically.
Use real-world examples when helpful.
"""

## Simple documentation lookup (tool)

In [10]:
def lookup_docs(query):
    docs = {
        "microservices": "Microservices split large apps into small independent services communicating via APIs.",
        "api": "An API allows systems to communicate using structured requests.",
        "streaming": "Streaming allows partial responses in real time instead of waiting for full completion."
    }
    
    for key in docs:
        if key in query.lower():
            return docs[key]
    
    return None

In [13]:
def chat(message, history, model):

    history = history or []

    messages = [{"role": "system", "content": system_prompt}]

    for msg in history:
        messages.append(msg)

    # TOOL USE
    tool_answer = lookup_docs(message)
    if tool_answer:
        return history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": f"📚 Tool:\n{tool_answer}"}
        ]

    messages.append({"role": "user", "content": message})

    stream = router.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    reply = ""
    updated_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""}
    ]

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        reply += delta
        updated_history[-1]["content"] = reply
        yield updated_history

In [14]:
demo = gr.ChatInterface(
    fn=chat,
    additional_inputs=gr.Dropdown(
        [
            "openai/gpt-4.1-mini",
            "anthropic/claude-3-haiku",
            "mistralai/mistral-7b-instruct"
        ],
        value="openai/gpt-4.1-mini",
        label="Choose Model"
    ),
    title="🧠 Technical AI Assistant"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.
